# RideFlow — 01 Landing (marketplace)

Generate FK-consistent synthetic RideFlow data with LakeLogic's simulator and load it into `LANDING.rideflow_*` (all VARCHAR) — the raw drop zone Bronze reads from. In production these would be fed by a managed connector / COPY INTO.


In [ ]:
# Trial task runs are headless — the internal PyPI mirror isn't available there, so we
# install LakeLogic + pure-python deps from wheels staged in Snowflake (@PIPELINE.WHEELS)
# with pip --no-index. (Interactive notebooks can `pip install lakelogic>=1.41.0` from the
# mirror instead.) The runtime already ships pandas/pydantic/pyyaml/snowflake-connector.
import os, subprocess, sys
from snowflake.snowpark.context import get_active_session
_session = get_active_session()
os.makedirs('/tmp/wl', exist_ok=True)
_session.file.get('@RIDEFLOW_DEV_DEMO.PIPELINE.WHEELS', '/tmp/wl')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                '--find-links', '/tmp/wl', '--no-deps', 'lakelogic', 'loguru', 'sqlglot'], check=True)


In [ ]:
import os, sys
# Resolve the repo root (folder containing domains_rideflow) from the Git-connected
# Workspace checkout, so runtime.* imports and relative contract paths both resolve.
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'domains_rideflow')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)

DB  = 'RIDEFLOW_DEV_DEMO'
SYS = 'domains_rideflow/marketplace/rideflow/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB   # contracts resolve {catalog} from this

from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection            # reuse THIS session (no password in the notebook)
# Give the session a current database + schema so the engine's TEMP tables have a
# home — interactive Workspace sessions have none set (headless NB objects default
# to their PIPELINE schema; the Workspace file does not).
session.sql(f'USE DATABASE {DB}').collect()
session.sql(f'USE SCHEMA {DB}.PIPELINE').collect()
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)
print('repo:', REPO)


In [ ]:
from runtime.landing import load_system_landing
loaded = load_system_landing(conn, DB, 'marketplace', 'rideflow', via_stage=True)
import pandas as pd
pd.DataFrame(loaded, columns=['landing_table','rows'])
